# Step 22.5 — Flash-style attention：数学相同，执行路径不同

## 先建立独立工作副本（以下只是命令参考，本次未执行）

在仓库根目录先同步母版：

```bash
git pull
```

再创建不覆盖已有文件的时间戳副本（仓库应已有 notebooks/work 目录）：

```bash
cp -n notebooks/101_step22_5_flash_style_attention.ipynb notebooks/work/101_step22_5_flash_style_attention_$(date +%Y%m%d_%H%M%S)_work.ipynb
```

只在工作副本中填写练习或未来运行。`cp -n` 在同名目标存在时不覆盖；同一秒再次操作也不会覆盖已有内容。当前母版是**材料初稿 0/2**，不是学习完成证明。

**执行边界：**所有代码单元 execution_count 为 null、outputs 为空。本次不执行 notebook、MLX 实验、测试、训练、验证、采样、语料读取/下载或 checkpoint 操作。下面的短实验是将来可观察的安排，不是已观察结果。MLX API 基线固定 v0.32.2。

## 1. 为什么需要这一节

普通 attention 容易显式构建 T×T score/probability 矩阵；长序列时，中间矩阵读写会成为巨大负担。Flash-style attention 用 tiling（分块）与在线归一化避免把完整矩阵反复写回较慢内存。它不把 dense attention 近似成稀疏注意力，而是改变同一数学运算的执行算法。

Fused kernel 把多步运算放在一个协调执行路径中，减少中间结果和调度开销。fast API 名称不是实际硬件已使用某一 FlashAttention 论文 kernel 的证明；具体选择受设备、shape、dtype、mask 和实现 heuristics 影响。

## 2. 先备数学与 Python / MLX 语义

对单行 scores sⱼ，stable softmax 输出为 o=Σexp(sⱼ−m)vⱼ / Σexp(sⱼ−m)，m=maxⱼ sⱼ。在线处理一个新块 b：m'=max(m,max s_b)，α=exp(m−m')；l'=αl+Σexp(s_b−m')；u'=αu+Σexp(s_b−m')v_b。最后 o=u/l。

这里 u 是未除分母的加权和，l 是归一化分母，m 是运行最大值。当新块最大值更大时，必须把旧 u/l 同时乘 α；只更新最大值不缩放旧累计会出错。本节只为无 mask、非空、有限分数推导单行 reference，避免全屏蔽行的 −Inf−(−Inf)。

## 3. 纸笔走一遍（讲解例子，不是练习答案）

scores=[0,ln2]，values=[2,5]，每块一个位置。第一块 m=0,l=1,u=2；第二块 m'=ln2，α=1/2，l'=1/2+1=1.5，u'=1+5=6，所以输出为 4。直接 softmax 加权为 (1/3)×2+(2/3)×5=4。

这种分块保持精确算术下的结果，浮点累加顺序不同仍会带来误差；“数学等价”不是“逐位一致”。

## 4. Shape tracing 与接口契约

Q/K/V (B,H,T,d) → 普通 score (B,H,T,T) → P 同 shape → output (B,H,T,d)。分块方案为每行维护 m/l 标量与 u 长 d 向量，并暂存有限 tile。Dense attention 的算术量仍为 O(T²d)，但不需要把完整 score/probability 驻留在外部内存。

本节先用单行 scores (T,) 和 values (T,dv) 观察在线归一化，再展示 MLX fast SDPA 与显式 causal reference 的接口；不写自定义 Metal kernel、不测峰值或速度。

In [ ]:
import math
import mlx.core as mx

# 本单元及后续所有单元均为未执行的教学源码。
# 仅在未来独立工作副本中按顺序运行；本次材料制作没有导入 MLX。

def online_attention_row(scores, values, tile_size):
    if scores.ndim != 1 or values.ndim != 2 or values.shape[0] != scores.shape[0]:
        raise ValueError("expect scores (T,) and values (T,dv)")
    if scores.shape[0] == 0 or type(tile_size) is not int or tile_size <= 0:
        raise ValueError("nonempty input and positive integer tile_size required")
    # 教学 reference：所有分数/values 有限；没有 causal 或 padding mask。
    m = None
    total = mx.array(0., dtype=mx.float32)
    weighted = mx.zeros((values.shape[1],), dtype=mx.float32)
    for start in range(0, scores.shape[0], tile_size):
        sb = scores[start:start + tile_size].astype(mx.float32)
        vb = values[start:start + tile_size].astype(mx.float32)
        block_max = mx.max(sb)
        new_m = block_max if m is None else mx.maximum(m, block_max)
        alpha = mx.array(0.) if m is None else mx.exp(m - new_m)
        weights = mx.exp(sb - new_m)
        total = alpha * total + mx.sum(weights)
        weighted = alpha * weighted + mx.sum(weights[:, None] * vb, axis=0)
        m = new_m
    return weighted / total

def dense_causal_attention(q, k, v):
    # Equal-length MHA only，非增量 cache 示例。
    T = q.shape[-2]
    scores = (q @ k.transpose(0, 1, 3, 2)) * (q.shape[-1] ** -0.5)
    allowed = mx.arange(T)[:, None] >= mx.arange(T)[None, :]
    return mx.softmax(mx.where(allowed, scores, float("-inf")), axis=-1) @ v

## 5. 短实验计划

先预测 shape 与趋势，再在未来工作副本中运行下一个单元。这里没有运行记录。若未来结果和推导不同，先检查输入、axis、dtype、mask、随机 key，再讨论数值误差；不要把一次随机输出当成分布证明。

In [ ]:
scores = mx.array([0., math.log(2)], dtype=mx.float32)
values = mx.array([[2.], [5.]], dtype=mx.float32)
online = online_attention_row(scores, values, tile_size=1)
dense = mx.softmax(scores, axis=-1) @ values
mx.eval(online, dense)
print("single-row outputs:", online.tolist(), dense.tolist())

q = mx.arange(24, dtype=mx.float32).reshape(1, 2, 3, 4) / 24
k, v = q + 0.1, q * 2
reference = dense_causal_attention(q, k, v)
fast = mx.fast.scaled_dot_product_attention(q, k, v, scale=0.5, mask="causal")
error = mx.max(mx.abs(reference - fast))
mx.eval(reference, fast, error)
print("future observed error:", error.item())

## 6. 常见误区与未来衔接

此 Python online reference 不是 FlashAttention kernel：MLX lazy graph 可能持有各块节点，Python loop 也有调度成本。不能用源码循环或未执行的样例声称节省实际峰值内存。真正性能评估需要控制设备、dtype、shape、预热、同步边界与 allocator 统计范围。

v0.32.2 的 `mask="causal"` 是 lower-right alignment（最后 query 对齐最后 key），方阵时与普通下三角一致。非方阵 prefill/decode 必须再次核对；array mask 为 broadcast-compatible Boolean 或 additive mask。`force_fused=False` 是默认，不强迫任何后端；强迫 fused 可能不支持或更慢，本课不设置。

课程范围决定：weight tying/RoPE/SwiGLU/GQA 是可选架构扩展；fast SDPA 是可选执行替换；KV cache 是可选推理状态设计。先保留已验收的最小模型作为基线，Step 23 的整理不应悄悄把这些概念示例混入核心实现。

## 7. 练习（不附答案）

1. 把三项 scores 分成两块，逐步列出 m/l/u 的更新。
2. 说明为什么每块分别 softmax 后直接把输出相加不是完整 softmax。
3. 设计未来公平比较 dense 与 fast 的记录表字段，区分理论复杂度、实际显存和延迟，不填写虚构数值。

请先用纸笔作答，再决定是否在工作副本中添加自己的代码。母版不填入参考答案。

## 8. 官方 API 阅读与完成边界

- [MLX v0.32.2 / python/src/fast.cpp](https://github.com/ml-explore/mlx/blob/v0.32.2/python/src/fast.cpp)

接口说明以固定版本源码为准。本次只有文本与源代码静态检查，没有运行通过声明。能够复述概念与完成练习后再进入下一节；材料覆盖范围不代表读者已经掌握。